In [1]:
import struct, sys
print(struct.calcsize("P") * 8, sys.version)   # 32 means 32-bit

64 3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]


In [1]:
# import dependencies
import sys
import json 
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, precision_recall_curve, fbeta_score)

import warnings
warnings.filterwarnings('ignore')

In [2]:
SEED = 42
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

In [ ]:
# load data
data = pd.read_csv("data/healthcare-dataset-stroke-data.csv")
print(f"Data shape: {data.shape}")

Data shape: (5110, 12)


In [4]:
data.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [5]:
# checking columns
print(data.columns.tolist())
print(data["work_type"].value_counts(), "\n")       # spelling

['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']
work_type
Private          2925
Self-employed     819
children          687
Govt_job          657
Never_worked       22
Name: count, dtype: int64 



In [6]:
# feature selections
y = data["stroke"]
X = data.drop(columns=["id", "stroke"])

num_cols = ["age", "avg_glucose_level", "bmi"]
cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
bin_cols = ["hypertension", "heart_disease"]    

In [7]:
# split stratified: 60% train, 20% validation, 20% test
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.40, stratify=y, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp,
                                            random_state=SEED)
for name, yy in [("train", y_tr), ("val", y_val), ("test", y_te)]:
    print(f"{name:5s} n={len(yy):5d}  stroke={yy.sum():4d} ({yy.mean():.3%})")

train n= 3066  stroke= 149 (4.860%)
val   n= 1022  stroke=  50 (4.892%)
test  n= 1022  stroke=  50 (4.892%)


## Preprocessing on TRAIN only
- median imputation (+ missing indicator) for numeric columns (bmi has NaNs)
- one-hot for categoricals, keeping 'Unknown' smoking as its own category
- no drop="first": keeps every dummy so SHAP labels stay readable

In [8]:
preprocessed = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median", add_indicator=True), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("bin", "passthrough", bin_cols),
    ],
    verbose_feature_names_out=False,
)
preprocessed.fit(X_tr)


def transform(Xdf):
    return pd.DataFrame(preprocessed.transform(Xdf), columns=preprocessed.get_feature_names_out(), index=Xdf.index)


X_tr_t, X_val_t, X_te_t = transform(X_tr), transform(X_val), transform(X_te)
print("\nTransformed features:", X_tr_t.shape[1])
print(X_tr_t.columns.tolist())
assert not X_tr_t.isna().any().any(), "NaNs remain after imputation"


Transformed features: 22
['age', 'avg_glucose_level', 'bmi', 'missingindicator_bmi', 'gender_Female', 'gender_Male', 'gender_Other', 'ever_married_No', 'ever_married_Yes', 'work_type_Govt_job', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'Residence_type_Rural', 'Residence_type_Urban', 'smoking_status_Unknown', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes', 'hypertension', 'heart_disease']


## Training blacbox model 1: XGBoost
- Imbalance strategy: scale_pos_weight (no resampling, so no synthetic rows)

In [9]:
spw = (y_tr == 0).sum() / (y_tr == 1).sum()
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw, eval_metric="aucpr",
    random_state=SEED, n_jobs=-1,
)
model.fit(X_tr_t, y_tr)
print("\nClasses:", model.classes_, "| scale_pos_weight:", round(spw, 2))


Classes: [0 1] | scale_pos_weight: 19.58


In [10]:
# choose decision threshold on VALIDATION data (maximise F2)
# F2 weights recall more than precision (missed strokes cost more).
p_val = model.predict_proba(X_val_t)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, p_val)
f2 = (5 * prec[:-1] * rec[:-1]) / np.maximum(4 * prec[:-1] + rec[:-1], 1e-12)
threshold = float(thr[np.argmax(f2)])
print(f"Selected threshold (max F2 on validation): {threshold:.4f}")

Selected threshold (max F2 on validation): 0.2485


In [11]:
# evaluate ONCE on the untouched test set
p_te = model.predict_proba(X_te_t)[:, 1]
y_hat = (p_te >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, y_hat).ravel()
metrics = {
    "threshold": threshold,
    "accuracy": accuracy_score(y_te, y_hat),
    "balanced_accuracy": balanced_accuracy_score(y_te, y_hat),
    "precision_stroke": precision_score(y_te, y_hat, zero_division=0),
    "recall_stroke": recall_score(y_te, y_hat),
    "f1_stroke": f1_score(y_te, y_hat),
    "f2_stroke": fbeta_score(y_te, y_hat, beta=2),
    "roc_auc": roc_auc_score(y_te, p_te),
    "average_precision": average_precision_score(y_te, p_te),
    "prevalence_test": float(y_te.mean()),
    "support_positive": int(y_te.sum()),
    "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
}
print("\nTEST METRICS")
for k, v in metrics.items():
    print(f"  {k:20s} {v:.4f}" if isinstance(v, float) else f"  {k:20s} {v}")
(OUT / "task2_metrics.json").write_text(json.dumps(metrics, indent=2))


TEST METRICS
  threshold            0.2485
  accuracy             0.6957
  balanced_accuracy    0.7452
  precision_stroke     0.1173
  recall_stroke        0.8000
  f1_stroke            0.2046
  f2_stroke            0.3697
  roc_auc              0.8207
  average_precision    0.2605
  prevalence_test      0.0489
  support_positive     50
  TN                   671
  FP                   301
  FN                   10
  TP                   40


447

## SHAP explainer + verification
- TreeExplainer on XGBoost explains the raw margin (LOG-ODDS) of class 1.
- Background: default tree-path-dependent (no background needed). 
- For interventional SHAP, pass data=shap.sample(X_tr_t, 100, random_state=SEED) and feature_perturbation="interventional".

In [12]:
explainer = shap.TreeExplainer(model)
sv = explainer(X_te_t)                      # Explanation: .values .base_values .data
print("\nSHAP values shape:", sv.values.shape, "(rows, features)")
assert sv.values.shape == X_te_t.shape
assert list(sv.feature_names) == list(X_te_t.columns), "feature order mismatch"

recon = sv.base_values + sv.values.sum(axis=1)                 # log-odds
margin = model.predict(X_te_t, output_margin=True)              # model's log-odds
resid = np.abs(recon - margin)
print(f"Base value (log-odds): {float(np.mean(sv.base_values)):.4f} "
      f"= probability {1/(1+np.exp(-float(np.mean(sv.base_values)))):.4f}")
print(f"Max |reconstruction - model margin|: {resid.max():.2e}  (tolerance 1e-4)")
assert resid.max() < 1e-4, "SHAP additivity check failed"
# Probability check (apply sigmoid to the LOG-ODDS sum, never add to a probability)
p_recon = 1 / (1 + np.exp(-recon))
print(f"Max |sigmoid(recon) - predict_proba|: {np.abs(p_recon - p_te).max():.2e}")

np.save(OUT / "shap_values_test.npy", sv.values)
X_te_t.to_csv(OUT / "X_test_transformed.csv")


SHAP values shape: (1022, 22) (rows, features)
Base value (log-odds): 0.4276 = probability 0.6053
Max |reconstruction - model margin|: 6.20e-06  (tolerance 1e-4)
Max |sigmoid(recon) - predict_proba|: 3.87e-07


## Task 3
- highest-scoring true positive in the test set. 
- Fall back to the highest-scoring case if there is none.

In [13]:
# local explanation for ONE held-out case
pos = np.where((y_te.values == 1) & (y_hat == 1))[0]
if len(pos):
    i = int(pos[np.argmax(p_te[pos])])
    case_type = "True positive"
else:
    i = int(np.argmax(p_te))
    case_type = "Highest-scoring case (no true positive)"
row_id = X_te_t.index[i]                                   # original row index
base = float(sv.base_values[i])
out_logit = float(sv.base_values[i] + sv.values[i].sum())

print(f"\nTask 3 case: test position {i}, original row {row_id}, {case_type}")
print(f"  actual={int(y_te.iloc[i])}, predicted={int(y_hat[i])}, threshold={threshold:.4f}")
print(f"  probability={p_te[i]:.4f}, log-odds output={out_logit:.4f}, base value={base:.4f}")

# Static force plot (for the Word report)
shap.force_plot(base, sv.values[i], X_te_t.iloc[i].round(2),
                matplotlib=True, show=False, figsize=(20, 3.5), text_rotation=20)
plt.savefig(OUT / "task3_force_plot.png", dpi=200, bbox_inches="tight")
plt.close()

# Interactive HTML (optional extra)
shap.save_html(str(OUT / "task3_force_plot.html"),
               shap.force_plot(base, sv.values[i], X_te_t.iloc[i]))

# Contribution table: top 3 by |SHAP|, plus the rest combined
contrib = pd.DataFrame({
    "feature": X_te_t.columns,
    "patient_value": X_te_t.iloc[i].values,
    "shap_logodds": sv.values[i],
})
contrib["direction"] = np.where(contrib["shap_logodds"] > 0, "Increases", "Decreases")
contrib = contrib.reindex(contrib["shap_logodds"].abs().sort_values(ascending=False).index)
top3, rest = contrib.head(3), contrib.iloc[3:]
table = pd.concat([top3, pd.DataFrame([{
    "feature": "Remaining features combined", "patient_value": np.nan,
    "shap_logodds": rest["shap_logodds"].sum(),
    "direction": "Increases" if rest["shap_logodds"].sum() > 0 else "Decreases",
}])], ignore_index=True)
print("\nTask 3 contribution table (log-odds scale):")
print(table.round(4).to_string(index=False))
table.to_csv(OUT / "task3_contributions.csv", index=False)
print("\nAll positive-direction features:", contrib[contrib.shap_logodds > 0].head(3)["feature"].tolist())
print("All negative-direction features:", contrib[contrib.shap_logodds < 0].head(3)["feature"].tolist())


Task 3 case: test position 34, original row 61, True positive
  actual=1, predicted=1, threshold=0.2485
  probability=0.9624, log-odds output=3.2416, base value=0.4276

Task 3 contribution table (log-odds scale):
                    feature  patient_value  shap_logodds direction
                        age           80.0        2.3368 Increases
               hypertension            1.0        0.4636 Increases
          avg_glucose_level           74.9        0.2376 Increases
Remaining features combined            NaN       -0.2241 Decreases

All positive-direction features: ['age', 'hypertension', 'avg_glucose_level']
All negative-direction features: ['bmi', 'smoking_status_never smoked', 'missingindicator_bmi']


## Reproducibility report

In [14]:
env = {
    "python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
    "scikit-learn": sklearn.__version__, "xgboost": xgb.__version__, "shap": shap.__version__,
    "seed": SEED, "split": "60/20/20 stratified",
    "train_idx_first5": X_tr.index[:5].tolist(),
}
(OUT / "environment.json").write_text(json.dumps(env, indent=2))
print("\nSaved outputs to", OUT.resolve())


Saved outputs to /Users/alyani/Desktop/grad school/xai-series-master/outputs
